# Notebook 2: Premier League player market valueGive it a player and a season of output, get back what they are worth.This one also feeds back into notebook 1. Market value is a better measure ofhow much a player matters than minutes and goals are, so once this works itdrops straight into the news branch and makes that smarter too.### Two things that decide whether this works**The target has to be logged.** Values run from roughly 25,000 euros to over200 million. A model trained on raw euros spends everything it has on thehandful of superstars and cannot tell a 500k player from a 5m one, which is afivefold difference that matters enormously to a real club.**Last year's value is a trap.** It is by far the strongest single predictor,because Transfermarkt's numbers move slowly and are anchored to their ownhistory. Include it and you get a model that scores brilliantly and has learnednothing about football. We train both versions and compare them, because thegap between them is the actual finding.### What good looks likeMarket value is a crowd estimate, not a fact, so there is no true number to hit.Landing within 25 percent of Transfermarkt on most players is solid. Anythingclaiming near-perfect accuracy has leaked the previous valuation.

In [ ]:
import sysfrom pathlib import PathROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT))import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport torchimport torch.nn as nnfrom sklearn.linear_model import Ridgefrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import mean_absolute_errorfrom src.config import INTERIM_DIR, PROCESSED_DIR, MODELS_DIRfrom src import market as Mfrom src import features as Fpd.set_option("display.width", 140)pd.set_option("display.max_columns", 50)plt.rcParams["figure.figsize"] = (10, 4)SEED = 42np.random.seed(SEED)torch.manual_seed(SEED)print("ready")

## 1. Build the datasetEverything comes from Transfermarkt's own tables, joined on `player_id`. Noname matching anywhere in the core pipeline, which means no silent row loss.The valuation for each player season is the one Transfermarkt published closestto the end of that season, found with a backward-only `merge_asof` so it cannever pick up a number published afterwards.The Elo table from notebook 1 comes in as club strength. Same output at a betterclub is worth more, and this is how the model learns that.

In [ ]:
# Club strength from notebook 1.feat_path = PROCESSED_DIR / "matches_features.csv"if feat_path.exists():    matches_feat = pd.read_csv(feat_path, parse_dates=["date"])    club_elo = F.final_elo_table(matches_feat)    print(f"club Elo for {len(club_elo)} clubs")else:    club_elo = None    print("No matches_features.csv. Run notebook 1 first for the Elo feature.")

In [ ]:
# Optional xG from FBref. Skipped cleanly if the join rate is poor.fbref_path = INTERIM_DIR / "players_shooting.csv"fbref = pd.read_csv(fbref_path, low_memory=False) if fbref_path.exists() else Nonedf = M.build(first_season=2012, club_elo=club_elo, fbref=fbref)print(f"\n{len(df):,} player seasons")print(f"{df['player'].nunique():,} distinct players")print(f"seasons {df['season_start'].min()} to {df['season_start'].max()}")df[["player", "club", "season_start", "age", "minutes", "goals",    "assists", "market_value_eur"]].head(10)

In [ ]:
# What does the target actually look like?fig, axes = plt.subplots(1, 2, figsize=(11, 4))axes[0].hist(df["market_value_eur"] / 1e6, bins=60, color="#1565c0")axes[0].set_title("market value, raw"); axes[0].set_xlabel("EUR millions")axes[1].hist(df["log_value"], bins=60, color="#2e7d32")axes[1].set_title("market value, logged"); axes[1].set_xlabel("log EUR")plt.tight_layout(); plt.show()print("The left panel is why we log it. Almost every player is crushed into")print("the first bar while a few superstars stretch the axis to the right.")print("The right panel is something a model can actually learn from.\n")print(df["market_value_eur"].describe().apply(M.euros).to_string())

### The age curveWorth plotting before modelling, because it explains why `age_squared` is inthe feature list. Value is not a straight line in age: it climbs through theearly twenties, peaks somewhere around 24 to 26, and falls off steeply after29. A linear term alone cannot represent that shape.

In [ ]:
by_age = df[df["age"].between(17, 38)].groupby(df["age"].round())["market_value_eur"]summary = by_age.median()ax = summary.plot(marker="o", color="#c62828")ax.set_xlabel("age"); ax.set_ylabel("median market value (EUR)")ax.set_title("Value by age")plt.tight_layout(); plt.show()peak = summary.idxmax()print(f"Median value peaks at age {peak:.0f} in this data.")

## 2. SplitBy season again, for the same reason as notebook 1. A random split would letthe model see a player's 2025 valuation while training on their 2024 one, andsince Transfermarkt values move slowly that is close to handing it the answer.

In [ ]:
TEST_SEASON = int(df["season_start"].max())VAL_SEASONS = [TEST_SEASON - 2, TEST_SEASON - 1]train = df[df["season_start"] < min(VAL_SEASONS)]val   = df[df["season_start"].isin(VAL_SEASONS)]test  = df[df["season_start"] == TEST_SEASON]print(f"train {len(train):,}  val {len(val):,}  test {len(test):,}")print(f"test season: {TEST_SEASON}/{(TEST_SEASON+1)%100:02d}")

In [ ]:
def prepare(include_prev_value):    X_all, cols = M.feature_matrix(df, include_prev_value=include_prev_value)    med = X_all.loc[train.index].median()    X_all = X_all.fillna(med)    out = {}    for name, part in (("train", train), ("val", val), ("test", test)):        out[name] = (X_all.loc[part.index].to_numpy(dtype=np.float32),                     part["log_value"].to_numpy(dtype=np.float32))    return out, colshonest, cols_honest = prepare(include_prev_value=False)leaky,  cols_leaky  = prepare(include_prev_value=True)print(f"without previous value: {len(cols_honest)} features")print(f"with previous value:    {len(cols_leaky)} features")

## 3. BaselinesTwo of them, and the first is deliberately stupid. Predicting the median valuefor a player's position and age bracket is what you could do with a lookuptable and no model at all. Anything that cannot beat it is not earning itskeep.

In [ ]:
# Lookup table baseline: median log value by position and age band.tr = train.copy()tr["age_band"] = pd.cut(tr["age"], [15, 21, 24, 27, 30, 45])lookup = tr.groupby(["pos_group", "age_band"], observed=True)["log_value"].median()overall = tr["log_value"].median()te = test.copy()te["age_band"] = pd.cut(te["age"], [15, 21, 24, 27, 30, 45])naive = [lookup.get((p, a), overall)         for p, a in zip(te["pos_group"], te["age_band"])]BASE = M.error_report(test["log_value"].to_numpy(), np.array(naive))print("Position and age lookup table")for k, v in BASE.items():    print(f"  {k}: {v:.3f}")

In [ ]:
def fit_ridge(data, label):    (X_tr, y_tr), (X_te, y_te) = data["train"], data["test"]    sc = StandardScaler().fit(X_tr)    model = Ridge(alpha=1.0).fit(sc.transform(X_tr), y_tr)    pred = model.predict(sc.transform(X_te))    rep = M.error_report(y_te, pred)    print(f"{label}:  median error {rep['median_ape']:.1%}   "          f"within 25% {rep['within_25pct']:.1%}   log R2 {rep['log_r2']:.3f}")    return repprint("Ridge regression")RIDGE_HONEST = fit_ridge(honest, "  without previous value")RIDGE_LEAKY  = fit_ridge(leaky,  "  with previous value   ")

## 4. Gradient boostingOn tabular data this shape it is usually the model to beat, same as notebook 1.

In [ ]:
from xgboost import XGBRegressordef fit_xgb(data, label):    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = (        data["train"], data["val"], data["test"])    model = XGBRegressor(        n_estimators=1200, max_depth=5, learning_rate=0.03,        subsample=0.8, colsample_bytree=0.8,        objective="reg:squarederror", eval_metric="rmse",        early_stopping_rounds=50, random_state=SEED,    )    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)    pred = model.predict(X_te)    rep = M.error_report(y_te, pred)    print(f"{label}:  median error {rep['median_ape']:.1%}   "          f"within 25% {rep['within_25pct']:.1%}   log R2 {rep['log_r2']:.3f}   "          f"({model.best_iteration} trees)")    return model, pred, repprint("XGBoost")xgb_honest, pred_honest, XGB_HONEST = fit_xgb(honest, "  without previous value")xgb_leaky,  pred_leaky,  XGB_LEAKY  = fit_xgb(leaky,  "  with previous value   ")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))for ax, model, cols, title in [    (axes[0], xgb_honest, cols_honest, "without previous value"),    (axes[1], xgb_leaky,  cols_leaky,  "with previous value"),]:    imp = pd.Series(model.feature_importances_, index=cols).nlargest(10)    imp.sort_values().plot(kind="barh", ax=ax, color="#1565c0")    ax.set_title(title)plt.tight_layout(); plt.show()print("Look at the right panel. If log_value_prev dwarfs everything else,")print("that model is mostly copying last year's number forward.")

## 5. Neural networkRegression this time, so one output and mean squared error rather than threeclasses and cross entropy. Same small architecture for the same reason: a fewthousand rows does not support anything deep.

In [ ]:
class ValueNet(nn.Module):    def __init__(self, n_features, hidden=(128, 64), dropout=0.2):        super().__init__()        h1, h2 = hidden        self.net = nn.Sequential(            nn.Linear(n_features, h1), nn.BatchNorm1d(h1), nn.ReLU(),            nn.Dropout(dropout),            nn.Linear(h1, h2), nn.BatchNorm1d(h2), nn.ReLU(),            nn.Dropout(dropout),            nn.Linear(h2, 1),        )    def forward(self, x):        return self.net(x).squeeze(-1)def train_valuenet(data, epochs=400, lr=1e-3, patience=40, verbose=True):    (X_tr, y_tr), (X_va, y_va), _ = data["train"], data["val"], data["test"]    sc = StandardScaler().fit(X_tr)    Xtr = torch.tensor(sc.transform(X_tr), dtype=torch.float32)    Xva = torch.tensor(sc.transform(X_va), dtype=torch.float32)    ytr = torch.tensor(y_tr, dtype=torch.float32)    yva = torch.tensor(y_va, dtype=torch.float32)    model = ValueNet(Xtr.shape[1])    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)    loss_fn = nn.MSELoss()    loader = torch.utils.data.DataLoader(        torch.utils.data.TensorDataset(Xtr, ytr), batch_size=128, shuffle=True)    best, best_state, wait, history = np.inf, None, 0, []    for epoch in range(epochs):        model.train()        for xb, yb in loader:            opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()        model.eval()        with torch.no_grad():            tr_l = loss_fn(model(Xtr), ytr).item()            va_l = loss_fn(model(Xva), yva).item()        history.append((tr_l, va_l))        if va_l < best - 1e-5:            best, wait = va_l, 0            best_state = {k: v.clone() for k, v in model.state_dict().items()}        else:            wait += 1            if wait >= patience:                if verbose:                    print(f"  early stop at epoch {epoch}, val MSE {best:.4f}")                break    model.load_state_dict(best_state); model.eval()    return model, sc, np.array(history)net, net_scaler, net_history = train_valuenet(honest)with torch.no_grad():    X_te = torch.tensor(net_scaler.transform(honest["test"][0]), dtype=torch.float32)    pred_net = net(X_te).numpy()NET_HONEST = M.error_report(honest["test"][1], pred_net)print(f"Neural network:  median error {NET_HONEST['median_ape']:.1%}   "      f"within 25% {NET_HONEST['within_25pct']:.1%}   "      f"log R2 {NET_HONEST['log_r2']:.3f}")

## 6. ResultsRead the two blocks separately. The honest block is the model that answers"what is this player worth based on who they are and what they did". Theanchored block answers "how will Transfermarkt revise their own number", whichis a much easier question and a much less interesting one.

In [ ]:
rows = [    {"model": "lookup table (pos + age)", "prev_value": "no", **BASE},    {"model": "ridge", "prev_value": "no", **RIDGE_HONEST},    {"model": "xgboost", "prev_value": "no", **XGB_HONEST},    {"model": "neural network", "prev_value": "no", **NET_HONEST},    {"model": "ridge", "prev_value": "yes", **RIDGE_LEAKY},    {"model": "xgboost", "prev_value": "yes", **XGB_LEAKY},]results = pd.DataFrame(rows)[    ["model", "prev_value", "median_ape", "within_25pct", "within_50pct",     "log_r2", "median_abs_error_eur"]]for c in ("median_ape", "within_25pct", "within_50pct"):    results[c] = (results[c] * 100).round(1)results["log_r2"] = results["log_r2"].round(3)results["median_abs_error_eur"] = results["median_abs_error_eur"].map(M.euros)results

In [ ]:
true = np.exp(honest["test"][1])pred = np.exp(pred_honest)fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))axes[0].scatter(true / 1e6, pred / 1e6, alpha=0.35, s=18, color="#1565c0")lim = max(true.max(), pred.max()) / 1e6axes[0].plot([0, lim], [0, lim], "k--")axes[0].set_xscale("log"); axes[0].set_yscale("log")axes[0].set_xlabel("actual (EUR m)"); axes[0].set_ylabel("predicted (EUR m)")axes[0].set_title("Predicted vs actual, log scale")resid = pred_honest - honest["test"][1]axes[1].scatter(true / 1e6, resid, alpha=0.35, s=18, color="#c62828")axes[1].axhline(0, color="k", ls="--")axes[1].set_xscale("log")axes[1].set_xlabel("actual (EUR m)"); axes[1].set_ylabel("log error")axes[1].set_title("Residuals")plt.tight_layout(); plt.show()print("A tilt in the residuals means the model compresses toward the middle:")print("it underestimates the expensive players and overestimates the cheap ones.")print("That is normal for a regression on a skewed target and worth naming.")

### Where it goes most wrongWorth reading the individual misses rather than only the aggregate. Large errorsusually cluster into recognisable groups: players who barely played, teenagersvalued on potential the model cannot see, and players whose value moved ontransfer speculation rather than performance.

In [ ]:
inspect = test.copy()inspect["predicted"] = np.exp(pred_honest)inspect["error_pct"] = (inspect["predicted"] - inspect["market_value_eur"]) \                       / inspect["market_value_eur"]show = ["player", "club", "age", "minutes", "goals", "assists",        "market_value_eur", "predicted", "error_pct"]print("Most overvalued by the model:")over = inspect.nlargest(8, "error_pct")[show].copy()for c in ("market_value_eur", "predicted"):    over[c] = over[c].map(M.euros)over["error_pct"] = (over["error_pct"] * 100).round(0)print(over.to_string(index=False))print("\nMost undervalued by the model:")under = inspect.nsmallest(8, "error_pct")[show].copy()for c in ("market_value_eur", "predicted"):    under[c] = under[c].map(M.euros)under["error_pct"] = (under["error_pct"] * 100).round(0)print(under.to_string(index=False))

In [ ]:
# Does playing time explain the misses?inspect["abs_err"] = inspect["error_pct"].abs()bands = pd.cut(inspect["minutes"], [0, 450, 900, 1800, 2700, 3420])by_band = inspect.groupby(bands, observed=True)["abs_err"].agg(["median", "count"])by_band["median"] = (by_band["median"] * 100).round(1)print("Median absolute error by minutes played\n")print(by_band.to_string())print()print("If error falls sharply as minutes rise, the model is fine and the")print("problem is that bench players have no signal to predict from.")

## 7. Save, and feed it back into notebook 1The valuation model is its own deliverable, but the more useful output rightnow is the value table itself.Notebook 1 scores player importance from minutes and goal involvements. Marketvalue is a better signal, because it already encodes reputation, potential andposition scarcity in a way that counting stats cannot. A 19 year old who played600 minutes might be the most valuable asset at the club, and minutes alonewould rate them as a fringe player.

In [ ]:
import jsonsc_final = StandardScaler().fit(honest["train"][0])torch.save(net.state_dict(), MODELS_DIR / "value_net.pt")np.savez(MODELS_DIR / "value_net_scaler.npz",         mean=net_scaler.mean_, scale=net_scaler.scale_)xgb_honest.save_model(MODELS_DIR / "value_xgb.json")meta = {    "features": cols_honest,    "test_season": f"{TEST_SEASON}/{(TEST_SEASON+1)%100:02d}",    "xgb": {k: round(v, 4) for k, v in XGB_HONEST.items()},    "net": {k: round(v, 4) for k, v in NET_HONEST.items()},    "note": "trained without previous valuation as a feature",}(MODELS_DIR / "value_meta.json").write_text(json.dumps(meta, indent=2))print("saved models")

In [ ]:
from src.importance import build_importance_from_valueimportance_v2 = build_importance_from_value(df)print(f"{len(importance_v2):,} player seasons with value-based importance\n")latest = int(importance_v2["season_start"].max())team = importance_v2[importance_v2["season_start"] == latest]["team"].iloc[0]print(f"{team}, {latest}/{(latest+1)%100:02d}")importance_v2[    (importance_v2["season_start"] == latest) & (importance_v2["team"] == team)].nlargest(15, "importance")[    ["player", "position", "minutes", "market_value_eur", "importance"]]

In [ ]:
# Compare the two importance signals on the same squad. Where they# disagree is where the new one earns its place.from src import importance as IMPold = IMP.load_importance()merged = old.merge(    importance_v2[["season_start", "team", "player_key", "importance"]],    on=["season_start", "team", "player_key"],    suffixes=("_minutes", "_value"), how="inner")merged["gap"] = merged["importance_value"] - merged["importance_minutes"]recent = merged[merged["season_start"] >= latest - 1]print("Rated much higher by market value than by minutes:")print(recent.nlargest(8, "gap")[    ["player", "team", "importance_minutes", "importance_value"]].round(2).to_string(index=False))print()print("These are usually young players and squad rotation at big clubs.")print("The news branch should treat them as more important than minutes")print("alone would suggest, which is exactly the upgrade.")

## Where this leaves usA valuation model, and a better importance signal that notebook 1 can useimmediately.### To wire it into notebook 1In the news branch, swap `IMP.build_importance()` for`build_importance_from_value(df)`. Everything downstream is unchanged, becauseboth produce the same columns. The injured-striker logic gets sharper for free.### Honest limitations**Transfermarkt values are crowd estimates, not facts.** We are modelling anopinion. When the model disagrees with Transfermarkt, neither side isautomatically wrong.**Bench players are close to unpredictable.** Under about 500 minutes there isbarely any performance signal, and the value is mostly reputation and potentialthat the features cannot see.**The data ends in mid 2026.** That project's collection pipeline is paused, so2026/27 squads are not covered and anyone signed recently is missing entirely.**Position grouping is coarse.** Four groups treats a holding midfielder and anattacking midfielder identically, and the market does not. `sub_position` is inthe dataset if you want to try finer grouping.### NextNotebook 3, player replacement, which is mostly built already. The featuretable here is the similarity space: standardise the per-90 columns, computecosine distance within position group, and filter by value and age.